In [37]:
import pathlib

import numpy as np
import scipy.sparse as sp
import scipy.io
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
import networkx as nx
import utils.preprocess
from sklearn.model_selection import train_test_split

## load

In [38]:
save_prefix = r'./data/IMDB_L_processed/graph_split/'
# save_prefix = r'D:\OneDrive\PhD\毕业\OHNN\demo_data\DBLP_processed/'
# read_perfix = 'demo_data/'
read_perfix = r'./data/IMDB_L/'

In [39]:
node = pd.read_csv(read_perfix + 'node.dat', sep = '\t\t', header=None)
link = pd.read_csv(read_perfix + 'link.dat', sep = '\t', header=None)
meta = pd.read_csv(read_perfix + 'meta.dat', sep = '\t', header=None)
label = pd.read_csv(read_perfix + 'label.dat', sep = '\t', header=None)
label_test = pd.read_csv(read_perfix + 'label.dat.test', sep = '\t', header=None)

d:\anaconda3\envs\ohnn\lib\site-packages\pandas\util\_decorators.py:311: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  return func(*args, **kwargs)


In [40]:
meta

,0
0,{
1,"""Node Total"": 21420,"
2,"""Node Type_0"": 4932,"
3,"""Node Type_1"": 2393,"
4,"""Node Type_2"": 6124,"
5,"""Node Type_3"": ,"
6,"""Edge Total"": 86642,"
7,"""Edge Type_0"": 4932,"
8,"""Edge Type_1"": 4932,"
9,"""Edge Type_2"": 14779,"


In [41]:
num_ntypes = 4

In [42]:
link.head(11)

,0,1,2,3
0,0,5851,0,1
1,5851,0,1,1
2,0,8129,2,1
3,8129,0,3,1
4,0,10128,2,1
5,10128,0,3,1
6,0,13298,2,1
7,13298,0,3,1
8,0,18753,4,1
9,18753,0,5,1


In [43]:
np.unique(link[2])

array([0, 1, 2, 3, 4, 5], dtype=int64)

## label

In [44]:
label.shape

(1371, 4)

In [45]:
label_test.shape

(3202, 4)

In [46]:
1090+1384+1837+1135+2517

7963

In [47]:
1371+3202

4573

In [48]:
label.head(10)


,0,1,2,3
0,1919,Unaccompanied_Minors,0,"2,0"
1,4635,Groove,0,4
2,3687,The_Forsaken,0,1
3,1662,Splice,0,4
4,4040,The_Deported,0,2
5,745,Up_Close_&_Personal,0,"4,0"
6,3182,The_Oxford_Murders,0,1
7,3123,"Run,_Fatboy,_Run",0,"2,0"
8,1776,Joyful_Noise,0,2
9,3107,Sleepover,0,"2,0"


In [49]:
[np.unique(label[p]) for p in label.columns]

[array([   3,    5,    6, ..., 4916, 4923, 4925], dtype=int64),
 array(['10_Cloverfield_Lane', '10th_&_Wolf', '12_Rounds', ..., 'xXx',
        'xXx:_State_of_the_Union', 'Æon_Flux'], dtype=object),
 array([0], dtype=int64),
 array(['0', '0,1', '1', '2', '2,0', '2,0,1', '2,1', '2,4', '2,4,0',
        '2,4,0,1', '2,4,1', '3', '3,0', '3,0,1', '3,1', '3,2', '3,2,0',
        '3,2,0,1', '3,2,1', '3,2,4', '3,2,4,0,1', '3,2,4,1', '3,4',
        '3,4,0', '3,4,0,1', '3,4,1', '4', '4,0', '4,0,1', '4,1'],
       dtype=object)]

In [50]:
label[[0,3]]

,0,3
0,1919,"2,0"
1,4635,4
2,3687,1
3,1662,4
4,4040,2
...,...,...
1366,4753,"3,4"
1367,507,"4,1"
1368,3253,4
1369,3983,2


## type_mask

In [51]:
edge0 = link[link[2] == 0]
edge0.shape #(4932, 4)
edge1 = link[link[2] == 1]
edge1.shape #(4932, 4)
edge2 = link[link[2] == 2]
edge2.shape #(14779, 4)
edge3 = link[link[2] == 3]
edge3.shape #(14779, 4)
edge4 = link[link[2] == 4]
edge4.shape #(23610, 4)
edge5 = link[link[2] == 5]
edge5.shape #(23610, 4)

(23610, 4)

In [97]:
edge5

,0,1,2,3
9,18753,0,5,1
11,20453,0,5,1
13,16336,0,5,1
15,16601,0,5,1
17,16222,0,5,1
...,...,...,...,...
86633,18997,4931,5,1
86635,20080,4931,5,1
86637,14232,4931,5,1
86639,19002,4931,5,1


In [52]:
tmp = [max(link[link[2] == p][0]) for p in range(6)]
tmp

[4931, 7324, 4931, 13448, 4931, 21419]

In [53]:
tmp2 = [min(link[link[2] == p][0]) for p in range(6)]
tmp2

[0, 4932, 0, 7325, 0, 13449]

In [54]:
tmp3 = [np.unique(link[link[2] == p][0]).shape for p in range(6)]
tmp3

[(4932,), (2393,), (4932,), (6124,), (4788,), (7971,)]

In [55]:
raw_nums = [4932, 2393, 6124, 7971]
nums = sum(raw_nums)
print(raw_nums)
print(nums)

prefix_operator = np.ones((num_ntypes+1, num_ntypes))
prefix_operator = np.tril(prefix_operator, k=-1)   # 下三角矩阵
prefix_operator = prefix_operator.dot(raw_nums).astype(int)
print(prefix_operator)

# 0 for movies 4661, 1 for directors 2270, 2 for actors 5841
type_mask = np.zeros(nums,dtype=int)
for i in range(num_ntypes):
    type_mask[prefix_operator[i]:prefix_operator[i+1]] = i

np.save(save_prefix + 'prefix_operator.npy',prefix_operator)
np.save(save_prefix + 'type_mask.npy',type_mask)

[4932, 2393, 6124, 7971]
21420
[    0  4932  7325 13449 21420]


## adjM

In [56]:
adjM = sp.lil_matrix((nums,nums)) # 21420
for head,tail,_,__ in link.values:
    adjM[head,tail] = 1

scipy.sparse.save_npz(save_prefix + 'adjM.npz', adjM.tocsr())
# lil matrix cost 4s on 3700x platform

## edges for MHGCN

In [100]:
edges = [sp.lil_matrix((nums,nums)) for _ in range(6)] # 6 edges
for i in range(6):
    edge = link[link[2] == i]
    edges[i][edge[0],edge[1]] = 1

scipy.io.savemat(save_prefix + 'A.mat', {'edges': [edge.tocsr() for edge in edges]})
edges

[<21420x21420 sparse matrix of type '<class 'numpy.float64'>'
 	with 4932 stored elements in List of Lists format>,
 <21420x21420 sparse matrix of type '<class 'numpy.float64'>'
 	with 4932 stored elements in List of Lists format>,
 <21420x21420 sparse matrix of type '<class 'numpy.float64'>'
 	with 14779 stored elements in List of Lists format>,
 <21420x21420 sparse matrix of type '<class 'numpy.float64'>'
 	with 14779 stored elements in List of Lists format>,
 <21420x21420 sparse matrix of type '<class 'numpy.float64'>'
 	with 23610 stored elements in List of Lists format>,
 <21420x21420 sparse matrix of type '<class 'numpy.float64'>'
 	with 23610 stored elements in List of Lists format>]

## ontology

In [57]:
prefix_operator = np.load(save_prefix + 'prefix_operator.npy')
type_mask = np.load(save_prefix + 'type_mask.npy')
adjM = scipy.sparse.load_npz(save_prefix + 'adjM.npz')
ontology = {
    'stem':[1,0,2],
    'branch':{0:[0,3]}
}
ontology_pairs = [[0,1],[0,2],[0,3]]

In [58]:
link_intances = utils.preprocess.get_intances_by_pair(adjM, type_mask, ontology, prefix_operator)
#print(link_intances)
print('=======done=======')

# cost about 0s with sparse matrix csr 
# nodes 21420
# stem 14779
# branch 23610

stem searching starts!
Wed Mar  6 19:50:58 2024, instances of (0, 2) have been found, counts is 14779.
merging path...

branch0 searching starts!
Wed Mar  6 19:50:58 2024, instances of (0, 3) have been found, counts is 23610.
merging path...

=======done=======


In [59]:
link_intances.keys()

dict_keys(['stem', 'branch'])

In [61]:
link_intances[''branch'][0'].shape

(14779, 3)

In [62]:
ontology = {
    'stem':[1,0,2],
    'branch':{0:[0,3]}
}

subgraphs = utils.preprocess.get_ontology_subgraphs_v3(ontology, link_intances)

subgraphs = subgraphs[subgraphs.columns.sort_values()]
print(len(subgraphs))
print(subgraphs)

# 0s
# 70790 subgraphs

70790
         0     1      2      3
0     3829  5354   7655  13898
1     3829  5354   7655  16610
2     3829  5354   7655  18907
3     3829  5354   7655  18962
4     3829  5354   7655  20641
...    ...   ...    ...    ...
5463  2357  7132  13095  15287
5464  2357  7132  13095  19277
5465  2357  7132  13095  19611
5466  2357  7132  13095  19663
5467  2357  7132  13095  19705

[70790 rows x 4 columns]


In [63]:
subgraphs.columns

Int64Index([0, 1, 2, 3], dtype='int64')

## save

In [66]:
# create the directories if they do not exist
for i in ['complete','incomplete']:
    pathlib.Path(save_prefix + '{}'.format(i)).mkdir(parents=True, exist_ok=True)

# save data 

# mapping of node to subgraphs

# mapping of node to node pairs 

# save schema
np.save(save_prefix + 'complete/ontology.npy', ontology) # schema
np.save(save_prefix + 'ontology_pairs.npy', ontology_pairs)
# subgraph table
np.save(save_prefix + 'complete/subgraphs.npy', subgraphs)
# all nodes adjacency matrix
scipy.sparse.save_npz(save_prefix + 'adjM.npz', adjM)
# all nodes features one-hot
for i in range(num_ntypes):
    scipy.sparse.save_npz(save_prefix + 'features_{}.npz'.format(i), scipy.sparse.eye(raw_nums[i]).tocsr())
# all nodes (authors, papers, terms and conferences) type labels
np.save(save_prefix + 'node_types.npy', type_mask)
# type prefix
np.save(save_prefix + 'prefix_operator.npy', prefix_operator)
# paper labels
# np.save(save_prefix + 'labels.npy', label[[0,3]]) # 5 class but not processed

## split

In [94]:
# subgraphs train/validation/test splits
rand_seed = 33333333
train_val_idx, test_idx = train_test_split(np.arange(adjM.shape[0]), test_size=0.08, random_state=rand_seed)
a = np.isin(subgraphs,test_idx)
a = a.sum(axis=1).astype('bool')
subgraphs_test = subgraphs[a]
subgraphs_tr_val = subgraphs[~a]
subgraphs[a].shape
print(subgraphs_test.shape[0]/len(subgraphs)) # 30% for test
train_idx, val_idx = train_test_split(train_val_idx, test_size=0.031, random_state=rand_seed)
b = np.isin(subgraphs_tr_val,val_idx)
b = b.sum(axis=1).astype('bool')
subgraphs_val = subgraphs_tr_val[b]
subgraphs_train = subgraphs_tr_val[~b]
subgraphs_tr_val[b].shape
print(subgraphs_val.shape[0]/len(subgraphs)) # 10% for val
print(len(subgraphs_train)/len(subgraphs)) # 60% for train

np.savez(save_prefix + 'complete/' + 'subgraphs_train_val_test.npz',
         subgraphs_train=subgraphs_train,
         subgraphs_val=subgraphs_val,
         subgraphs_test=subgraphs_test) # subgraph train&val&test
# node split
np.savez(save_prefix + 'complete/' + 'train_val_test_nodes.npz',
         train_nodes=train_idx,
         val_nodes=val_idx,
         test_nodes=test_idx) # nodes train&val&test

0.30759994349484393
0.0928238451758723
0.5995762113292838


## =============================================================